# ARIA Airborne Exploratory Data Analysis (EDA) with PySpark
**Author:** Brett Allen (brett.allen@gdit.com)

**Environment:**
```
   SageMaker Image: SparkAnalytics 3.0
  SageMaker Kernel: Glue PySpark
SageMaker Instance: ml.t3.medium (2 vCPU + 4 GiB @ $0.0416/hr) - EBS storage only (for HDFS)
```

In [ ]:
%iam_role arn:aws:iam::867344433302:role/endurasoft-GlueJobServiceRole
%profile default
%etl
%number_of_workers 2
%worker_type G.2X
%glue_version 3.0
%additional_python_modules "plotly-express,datashader,geopandas,shapely,dask[dataframe],folium"

## Imports

In [ ]:
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.window import Window
from pyspark import StorageLevel
from pyspark.context import SparkContext
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.stat import Correlation
from pyspark.mllib.linalg.distributed import RowMatrix
import pandas as pd
import geopandas as gpd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px
from colorcet import fire
import datashader.transfer_functions as tf
import datashader as ds
import boto3
import io

In [ ]:
%status

## Configurations

In [ ]:
# %matplotlib inline

In [ ]:
# Initialize spark context and glue context to create glue job for analysis
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [ ]:
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.parquet.enableVectorizedReader","false")

In [ ]:
s3_client = boto3.client('s3')

## Load the Data

### Track Points
Loading OpenSky Network data.
* Flight Recorder Data: `s3://endurasoft-dev-risk-framework/opensky-network/track-points/`

In [ ]:
track_points_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/opensky-network/track-points/year=*/month=*/day=*/hour=*/*.parquet")
track_points_df.printSchema()

In [ ]:
track_points_df.persist()

### UAS Sightings
Loading UAS sightings reports converted to tabular format.
* `s3://endurasoft-dev-risk-framework/datasets/uas_sightings_reports/zero_shot/uas_sightings.parquet`

In [ ]:
uas_sightings_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/datasets/uas_sightings_reports/zero_shot/uas_sightings.parquet")
uas_sightings_df.printSchema()

In [ ]:
uas_sightings_df.persist()

### UAS Facility Maps (Grids)
Loading UAS Facility Map (UASFM) grids data. This data should be fused with UAS sightings data and track points data for risk analysis.
* `s3://endurasoft-dev-risk-framework/datasets/uasfm_grids/FAA_UAS_FacilityMap_Data.parquet`

In [ ]:
uasfm_df = spark.read.parquet("s3://endurasoft-dev-risk-framework/datasets/uasfm_grids/FAA_UAS_FacilityMap_Data.parquet")
uasfm_df.printSchema()

In [ ]:
uasfm_df.persist()

## Exploratory Data Analysis (EDA)

In [ ]:
def clean_column_names(df):
    # Convert column names to lowercase and replace spaces, dots, and forward slashes with underscores
    return df.toDF(*[ re.sub(r'[ \.\/]+', '_', c.lower()) for c in df.columns ])

In [ ]:
# Create function to show more information on missing values
def get_missing_values(df: DataFrame) -> DataFrame:
    """
    Create a dataframe to represent the missing values in the original dataframe.

    Args:
        df (DataFrame): Original dataframe to identify missing values.

    Returns:
        DataFrame: New dataframe representing a report of missing values in original dataframe.
    """
    total_records = df.count()
    
    # For each column, calculate the total missing, available, ratio, and percentage of missing values
    missing_stats = []
    
    for column in df.columns:
        total_missing = df.select(F.count(F.when(F.col(column).isNull(), column)).alias("total_missing")).collect()[0][0]
        total_available = total_records - total_missing
        ratio_missing = total_missing / total_records
        percent_missing = round(ratio_missing * 100, 1)
        
        missing_stats.append({
            'column': column,
            'total_missing': total_missing,
            'total_available': total_available,
            'ratio_missing': round(ratio_missing, 4),
            'percent_missing': f'{percent_missing}%'
        })
    
    # Create a DataFrame from the list of dictionaries
    result_df = df.sql_ctx.createDataFrame(missing_stats)
    
    # Ensure correct column order
    return result_df.select(['column', 'total_missing', 'total_available', 'ratio_missing', 'percent_missing'])

In [ ]:
def get_unique_values(df):
    # NOTE: Can use approx_count_distinct (see https://stackoverflow.com/a/53764762)
    return df.agg(*(F.countDistinct(F.col(c)).alias(c) for c in df.columns))

### Track Points

In [ ]:
# Clean column names
track_points_df = clean_column_names(track_points_df)

In [ ]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in track_points_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    track_points_df = track_points_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [ ]:
track_points_df.persist()

In [ ]:
track_points_df.show(n=1, vertical=True, truncate=False)

#### Descriptive Statistics

In [ ]:
track_points_df.describe().show(n=len(track_points_df.columns), vertical=True, truncate=False)

#### Date Range and Total Records

In [ ]:
# Analyze date range for dataset
track_points_df = track_points_df.withColumn('ateventtime_timestamp', F.to_timestamp('ateventtime_timestamp'))

# Get the start and end dates
track_points_start_date = track_points_df.select(F.min("ateventtime_timestamp")).collect()[0][0]
track_points_end_date = track_points_df.select(F.max("ateventtime_timestamp")).collect()[0][0]

print("Start Date :", track_points_start_date)
print("End Date   :", track_points_end_date)

In [ ]:
track_points_date_diff = track_points_end_date - track_points_start_date
print(f'Northern California TRACON dataset spans {track_points_date_diff.days} days ({round(track_points_date_diff.days/365, 2)} years)')

In [ ]:
print(f'Total records for Northern California TRACON: {track_points_df.count():,}')

#### Analyze Nulls

In [ ]:
track_points_missing_values_report = get_missing_values(track_points_df)
track_points_missing_values_report.show(n=len(track_points_df.columns), truncate=False)

#### Analyze Uniqueness

In [ ]:
track_points_unique_counts = get_unique_values(track_points_df)
track_points_unique_counts.show(n=len(track_points_df.columns), vertical=True, truncate=False)

In [ ]:
track_points_unique_counts.toPandas()

**(TODO) Observation:**

Low variance columns (categorical):
* List columns here

#### Encode Categorical Features
**NOTE**: Need to encode categorical string features with low variance and target only numeric features prior to identifying important/primary features

In [ ]:
categorical_features = [
    'conflictangle',
    'aircraft_0_climbstatus',
    'aircraft_0_direction',
    'aircraft_0_aircrafttype',
    'aircraft_0_ifrvfrstatus',
    'aircraft_0_aircraftclass',
    'aircraft_0_enginetype',
    'aircraft_0_pilotsystem',
    'aircraft_1_climbstatus',
    'aircraft_1_direction',
    'aircraft_1_aircrafttype',
    'aircraft_1_ifrvfrstatus',
    'aircraft_1_aircraftclass',
    'aircraft_1_enginetype',
    'aircraft_1_pilotsystem',
]

encoded_features = [col + "_encoded" for col in categorical_features]

# Create a list of StringIndexer transformers
indexers = [StringIndexer(inputCol=col, outputCol=col + "_encoded") for col in categorical_features]

# Create a VectorAssembler to combine the indexed columns
assembler = VectorAssembler(inputCols=encoded_features, outputCol="features")

# Create a Pipeline to chain the transformers and assembler
pipeline = Pipeline(stages=indexers + [assembler])

# Fit and transform the DataFrame
indexer_model = pipeline.fit(track_points_df)
track_points_df_encoded = indexer_model.transform(track_points_df)

In [ ]:
track_points_df_encoded.select(*encoded_features).show(n=1, vertical=True, truncate=False)

In [ ]:
# Apply encoded changes
track_points_df = track_points_df_encoded

In [ ]:
# Drop features column that was used for indexer model
track_points_df = track_points_df.drop('features')

In [ ]:
track_points_df.show(n=1, vertical=True, truncate=False)

#### Identify Important/Primary Features

In [ ]:
track_points_df.dtypes

In [ ]:
numeric_cols = [c for c, t in track_points_df.dtypes if t in ('double', 'int', 'bigint', 'long', 'float')]
numeric_cols

In [ ]:
features = numeric_cols
target = 'eventscore'

# Create a VectorAssembler to combine features
assembler = VectorAssembler(inputCols=features, outputCol="rf_features", handleInvalid='skip')

# Create a RandomForestRegressor
rf = RandomForestRegressor(labelCol=target, featuresCol="rf_features", numTrees=100, maxBins=2000)

# Create a pipeline
pipeline = Pipeline(stages=[assembler, rf])

In [ ]:
# Split the data into training and testing sets
train_data, test_data = track_points_df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# Fit the random forest regressor model via pipeline
# TODO Need to analyze null data further and determine best approach for handling nulls (e.g., imputing null values or dropping rows with null values)
rf_model = pipeline.fit(train_data)

In [ ]:
# Make predictions on the test data
# predictions = rf_model.transform(test_data)

In [ ]:
# Evaluate the model
# evaluator = RegressionEvaluator(labelCol=target, predictionCol="prediction", metricName="rmse")
# rmse = evaluator.evaluate(predictions)
# print("Root Mean Squared Error (RMSE):", rmse)

In [ ]:
# Analyze feature importances
feature_importances = rf_model.stages[1].featureImportances
feature_importances

In [ ]:
# Map feature importances back to column names and build pandas dataframe, sorted by importance in descending order
feature_importance_dict = dict(zip(assembler.getInputCols(), feature_importances))
feature_importance_df = pd.DataFrame({'Feature': feature_importance_dict.keys(), 'Importance': feature_importance_dict.values()})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
pd.set_option('display.max_rows', 100)
feature_importance_df.round(6)

In [ ]:
f, ax = plt.subplots(figsize=(16, 8))
ax = sns.barplot(data=feature_importance_df, x='Importance', y='Feature')
ax.set_title("Visualize feature scores of the features", weight='bold')
ax.set_xlabel("Feature importance score")
ax.set_ylabel("Features")
plt.tight_layout()
plt.show()
%matplot plt

#### Analyze Correlations for Primary Features

In [ ]:
# Assemble features into a vector column
numeric_track_points_df = track_points_df.select(*numeric_cols)
assembler = VectorAssembler(inputCols=numeric_track_points_df.columns, outputCol="numeric_features", handleInvalid='skip')
numeric_track_points_df = assembler.transform(numeric_track_points_df)

# Calculate correlation matrix
corr_matrix = Correlation.corr(numeric_track_points_df, "numeric_features").head()[0].toArray()

In [ ]:
# Convert to pandas dataframe for plotting
corr_df = pd.DataFrame(corr_matrix, columns=numeric_track_points_df.columns[:-1], index=numeric_track_points_df.columns[:-1])

In [ ]:
corr_df

In [ ]:
# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(32, 16))
sns.heatmap(corr_df, annot=True, fmt='.1f', annot_kws={"fontsize": 5}, cmap="coolwarm")
plt.title('Northern California TRACON Correlation Matrix')
plt.tight_layout()
# plt.savefig('./track_points_numeric_features_correlation_matrix.png') # NOTE: savefig does not work in glue pyspark. Need to shift+right click on the image and save manually
plt.show()
%matplot plt

In [ ]:
corr_df.to_csv('s3://gdit-faa-datachallenge-proto/notebooks/aria_airborne_pyspark_eda/track_points_numeric_features_correlation_matrix.csv', index=False)

In [ ]:
# Reduce number of features and select only features of interest
features_of_interest = [
    'eventscore',
    'latitude',
    'longitude',
    'isinsideairspace',
    'isneartower',
    'timetocpainmillisec',
    'isleveloffevent',
    'coursedelta',
    'conflictangle_encoded',
    'ateventtime_truelateralnm',
    'ateventtime_estlateralatcpanm',
    'atestimatedcpatime_vertclosurerateftpermin',
    'atestimatedcpatime_lateralclosureratekt',
    'aircraft_0_altitudeinfeet',
    'aircraft_0_climbstatus_encoded',
    'aircraft_0_speedinknots',
    'aircraft_1_altitudeinfeet',
    'aircraft_1_climbstatus_encoded',
    'aircraft_1_speedinknots',
]

corr_df_foi = corr_df.loc[features_of_interest, :][features_of_interest]

# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(30, 12))
sns.heatmap(corr_df_foi, annot=True, fmt='.1f', annot_kws={"fontsize": 12}, cmap="coolwarm")
plt.xticks(rotation=70)
plt.title('Northern California TRACON Correlation Matrix (Features of Interest)', weight='bold')
plt.tight_layout()
plt.show()
%matplot plt

#### Analyze Covariance
Covariance measures the influence of change between features.

In [ ]:
numeric_track_points_df.columns

In [ ]:
vector_col = "cov_features"
assembler = VectorAssembler(inputCols=numeric_track_points_df.columns[:-1], outputCol=vector_col, handleInvalid="skip")
df_vector = assembler.transform(numeric_track_points_df).select(vector_col)

In [ ]:
# Must have single vector dtype
df_vector.dtypes

In [ ]:
row_matrix = RowMatrix(df_vector.rdd.map(list))
cov_matrix = row_matrix.computeCovariance()

In [ ]:
# Convert to pandas dataframe for plotting
cov_df = pd.DataFrame(cov_matrix.toArray(), columns=numeric_track_points_df.columns[:-1], index=numeric_track_points_df.columns[:-1])

In [ ]:
cov_df

In [ ]:
# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(32, 16))
sns.heatmap(cov_df, annot=True, fmt='.1g', annot_kws={"fontsize": 5}, cmap="coolwarm")
plt.title('Northern California TRACON Covariance Matrix')
plt.tight_layout()
plt.show()
%matplot plt

In [ ]:
cov_df_foi = cov_df.loc[features_of_interest, :][features_of_interest]

# Plot heatmap
# See https://repost.aws/questions/QULOKRL66TSWy-WY4F9kL25w/how-to-get-glue-interactive-session-notebook-to-show-matplotlib-plot
plt.clf() # Clear current figure
fig = plt.figure(figsize=(30, 12))
sns.heatmap(cov_df_foi, annot=True, fmt='.1g', annot_kws={"fontsize": 12}, cmap="coolwarm")
plt.xticks(rotation=70)
plt.title('Northern California TRACON Covariance Matrix (Features of Interest)', weight='bold')
plt.tight_layout()
plt.show()
%matplot plt

* **Positive covariance**: The two variables tend to move in the same direction. 
* **Negative covariance**: The two variables tend to move in opposite directions. 
* **Zero covariance**: The two elements do not vary together.

#### Analyze Events by Altitude Bins

In [ ]:
alt_bins = [0, 150, 200, 250, 500, 750, 1000, 2500, 5000, 10000, float("inf")]
alt_labels = ['<150', '150-200', '200-250', '250-500', '500-750', '750-1000', '1000-2500', '2500-5000', '5000-10000', '>10000']

In [ ]:
# Create the altitude bins for aircraft_0_altitudeinfeet and aircraft_1_altitudeinfeet
track_points_df = track_points_df.withColumn("aircraft_0_altitude_bin", 
                   F.when(F.col("aircraft_0_altitudeinfeet") < alt_bins[1], alt_labels[0])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[1]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[2]), alt_labels[1])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[2]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[3]), alt_labels[2])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[3]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[4]), alt_labels[3])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[4]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[5]), alt_labels[4])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[5]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[6]), alt_labels[5])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[6]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[7]), alt_labels[6])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[7]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[8]), alt_labels[7])
                   .when((F.col("aircraft_0_altitudeinfeet") >= alt_bins[8]) & (F.col("aircraft_0_altitudeinfeet") < alt_bins[9]), alt_labels[8])
                   .otherwise(alt_labels[9]))

track_points_df = track_points_df.withColumn("aircraft_1_altitude_bin", 
                   F.when(F.col("aircraft_1_altitudeinfeet") < alt_bins[1], alt_labels[0])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[1]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[2]), alt_labels[1])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[2]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[3]), alt_labels[2])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[3]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[4]), alt_labels[3])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[4]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[5]), alt_labels[4])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[5]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[6]), alt_labels[5])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[6]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[7]), alt_labels[6])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[7]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[8]), alt_labels[7])
                   .when((F.col("aircraft_1_altitudeinfeet") >= alt_bins[8]) & (F.col("aircraft_1_altitudeinfeet") < alt_bins[9]), alt_labels[8])
                   .otherwise(alt_labels[9]))

In [ ]:
# Analyze event score by aircraft_0_altitude_bin
print('='*100)
print(f'Northern California TRACON Events Analysis by "aircraft_0_altitude_bin"')
print('='*100)
track_points_df.groupBy('aircraft_0_altitude_bin').agg(
    F.count('eventscore').alias('event_count'),
    F.min('eventscore').alias('min_eventscore'),
    F.max('eventscore').alias('max_eventscore'),
    F.avg('eventscore').alias('average_eventscore'),
    F.stddev('eventscore').alias('stddev_eventscore'),
).orderBy(F.col('average_eventscore')).show()

In [ ]:
# Analyze event score by aircraft_1_altitude_bin
print('='*100)
print(f'Northern California TRACON Events Analysis by "aircraft_1_altitude_bin"')
print('='*100)
track_points_df.groupBy('aircraft_1_altitude_bin').agg(
    F.count('eventscore').alias('event_count'),
    F.min('eventscore').alias('min_eventscore'),
    F.max('eventscore').alias('max_eventscore'),
    F.avg('eventscore').alias('average_eventscore'),
    F.stddev('eventscore').alias('stddev_eventscore'),
).orderBy(F.col('average_eventscore')).show()

### Aggregate Analysis
Combine both tracons for aggregate analysis

In [ ]:
combined_df = spark.read.parquet('s3://gdit-faa-datachallenge-proto/converteddata/aria_airborne/*/*.parquet')

In [ ]:
# Clean column names
combined_df = clean_column_names(combined_df)

In [ ]:
# Convert boolean columns to integer
bool_cols = [col_name for col_name, col_type in combined_df.dtypes if col_type == "boolean"]

# Convert boolean columns to integers
for col_name in bool_cols:
    combined_df = combined_df.withColumn(col_name, F.col(col_name).cast(T.IntegerType()))

In [ ]:
combined_df.persist()

#### Analyze Events by Facility

In [ ]:
combined_df.groupBy('facility').count().show()

In [ ]:
# Average event score by facility
combined_df.groupBy('facility').agg(
    F.count('eventscore').alias('event_count'),
    F.min('eventscore').alias('min_eventscore'),
    F.max('eventscore').alias('max_eventscore'),
    F.avg('eventscore').alias('average_eventscore'),
    F.stddev('eventscore').alias('stddev_eventscore'),
).show()

#### Analyze Events by Geolocation

##### Clustered Heatmap (Folium)

In [ ]:
cell_size = 0.007

df_grid = combined_df.withColumn("grid_lat", F.floor(combined_df["latitude"] / cell_size) * cell_size) \
             .withColumn("grid_lon", F.floor(combined_df["longitude"] / cell_size) * cell_size) \
             .groupBy("grid_lat", "grid_lon") \
             .agg(F.count("*").alias("point_count"))

In [ ]:
df_grid_pandas = df_grid.toPandas()

In [ ]:
df_grid_pandas

In [ ]:
map_center = [df_grid_pandas["grid_lat"].mean(), df_grid_pandas["grid_lon"].mean()]
m = folium.Map(location=map_center, zoom_start=10)

# Create the heatmap layer
HeatMap(
    data=df_grid_pandas[["grid_lat", "grid_lon", "point_count"]].values.tolist(),
    radius=15, 
    blur=10
).add_to(m)
m

In [ ]:
# s3://gdit-faa-datachallenge-proto/notebooks/aria_airborne_pyspark_eda/
map_filename = 'aria_airborne_events_geolocation.html'
buf = io.BytesIO(m._repr_html_().encode('utf-8'))
s3_client.put_object(Bucket='gdit-faa-datachallenge-proto', Key=f'notebooks/aria_airborne_pyspark_eda/{map_filename}', Body=buf)
buf.close()

##### Shader-based Point Density Heatmap (Plotly)

In [ ]:
# Apply moving average to reduce density of track points
# uniqueid
# latitude
# longitude
# ateventtime_timestamp - 2022-05-13T17:36:42.971Z         

In [ ]:
# ateventtime_timestamp - 2022-05-13T17:36:42.971Z -> format string: "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'" (can be inferred by pyspark)
combined_df = combined_df.withColumn('ateventtime_timestamp', F.to_timestamp('ateventtime_timestamp'))

In [ ]:
combined_df.select('ateventtime_timestamp').dtypes

In [ ]:
# Define the window specification
# window_spec = Window.partitionBy("uniqueid").orderBy("ateventtime_timestamp").rowsBetween(-1, 1)

# # Calculate the moving average for latitude and longitude
# windowed_df = combined_df.withColumn("avg_latitude", F.avg("latitude").over(window_spec))
# windowed_df = windowed_df.withColumn("avg_longitude", F.avg("longitude").over(window_spec))

# # Select the desired columns (including the moving averages)
# windowed_df = windowed_df.select("uniqueid", "ateventtime_timestamp", "avg_latitude", "avg_longitude")

# # Drop duplicates (optional, to further reduce density)
# mav_df = windowed_df.dropDuplicates(["uniqueid", "avg_latitude", "avg_longitude"])

# mav_df.show(n=1, vertical=True, truncate=False)

In [ ]:
# Determine average time diff in seconds between events; this will help determine reduction ratio
# Calculate the time difference between consecutive records
window = Window.orderBy(F.col("ateventtime_timestamp"))
combined_df = combined_df.withColumn("prev_timestamp", F.lag("ateventtime_timestamp").over(window))
combined_df = combined_df.withColumn("time_diff", F.unix_timestamp("ateventtime_timestamp") - F.unix_timestamp("prev_timestamp"))

# Calculate the average time difference in seconds
avg_time_diff = combined_df.agg(F.avg("time_diff")).collect()[0][0]

print("Average time difference in seconds:", avg_time_diff)

In [ ]:
# Create year, month, day, hour, minute, seconds columns
combined_df = combined_df.withColumn("year", F.year("ateventtime_timestamp")) \
                         .withColumn("month", F.month("ateventtime_timestamp")) \
                         .withColumn("day", F.dayofmonth("ateventtime_timestamp")) \
                         .withColumn("hour", F.hour("ateventtime_timestamp")) \
                         .withColumn("minute", F.minute("ateventtime_timestamp")) \
                         .withColumn("second", F.second("ateventtime_timestamp"))

In [ ]:
# Apply date range based windowing based on https://stackoverflow.com/a/45824339
# NOTE: This will take a while to run depending on the size of the dataset
# Function to calculate number of seconds from number of days
days_to_seconds = lambda i: i * 86400

# Create window by casting timestamp to long (number of seconds)
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.rangeBetween.html
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.rowsBetween.html
window_spec = Window.partitionBy("uniqueid").orderBy(F.col("ateventtime_timestamp").cast('long')).rangeBetween(-days_to_seconds(1), 0)
# window_spec = Window.orderBy(F.col('ateventtime_timestamp')).rowsBetween(-1, 0)
# window_spec = Window.partitionBy("facility").orderBy(F.col("ateventtime_timestamp")).rowsBetween(Window.currentRow, 1)

# Calculate the moving average for latitude and longitude
windowed_df = combined_df.withColumn("avg_latitude", F.avg("latitude").over(window_spec))
windowed_df = windowed_df.withColumn("avg_longitude", F.avg("longitude").over(window_spec))

# Calculate the moving average for event score
# windowed_df = combined_df.withColumn("avg_eventscore", F.avg("eventscore").over(window_spec))

# Drop duplicates to further reduce density
mav_df = windowed_df.dropDuplicates(["uniqueid", "avg_latitude", "avg_longitude"])
# mav_df = windowed_df.dropDuplicates(["avg_eventscore"])
# mav_df.show(n=1, vertical=True, truncate=False)

In [ ]:
# Analyze density reduction with moving average
# total starting records = 3,590,562
#   reduced to 3,039,839 with rowsBetween(-2, 0)
#   reduced to 2,897,022 with rowsBetween(-1, 0)
#   reduced to 2,396,445 with rowsBetween(Window.currentRow, 1)
f'{combined_df.count():,}', f'{mav_df.count():,}'

In [ ]:
reduced_df = combined_df.withColumn('trunc_timestamp', F.date_trunc('second', F.col('ateventtime_timestamp')))\
                        .groupBy('trunc_timestamp')\
                        .agg(*[F.first(col).alias(col) for col in combined_df.columns])\
                        .select(*combined_df.columns)\
                        .orderBy(F.col('ateventtime_timestamp').asc())

In [ ]:
# Analyze density reduction with date truncation
f'{combined_df.count():,}', f'{reduced_df.count():,}'

In [ ]:
sampled_df = combined_df.sample(fraction=0.1, seed=42)

In [ ]:
# Analyze density reduction with sampling
f'{combined_df.count():,}', f'{sampled_df.count():,}'

In [ ]:
sampled_df_pandas = sampled_df.toPandas()

In [ ]:
# Plot Latitude & Longitude on a map using plotly and datashader - https://plotly.com/python/datashader/
cvs = ds.Canvas(plot_width=1000, plot_height=1000)
agg = cvs.points(sampled_df_pandas, x='longitude', y='latitude')
coords_lat, coords_lon = agg.coords['latitude'].values, agg.coords['longitude'].values

# Corners of the image, which need to be passed to mapbox
coordinates = [[coords_lon[0], coords_lat[0]],
               [coords_lon[-1], coords_lat[0]],
               [coords_lon[-1], coords_lat[-1]],
               [coords_lon[0], coords_lat[-1]]]

img = tf.shade(agg, cmap=fire)[::-1].to_pil()

# Trick to create rapidly a figure with mapbox axes
fig = px.scatter_mapbox(
    sampled_df_pandas[:1], # First row, all columns
    lat='latitude', 
    lon='longitude',
    opacity=0.0, # Hidden
    zoom=10,
    height=600,
    width=1600,
    center={'lat': 36.778259, 'lon': -119.417931}, # California
)

# Add the datashader image as a mapbox layer image
fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_layers = [
        {
            "sourcetype": "image",
            "source": img,
            "coordinates": coordinates
        }
    ],
    margin=dict(l = 0, r = 0, t = 0, b = 0),
)
fig.show()

In [ ]:
# Save the plotly heatmap
string_buf = io.StringIO()
fig.write_html(string_buf)
string_buf.seek(0)

string_data = string_buf.getvalue()
bytes_buf = io.BytesIO(string_data.encode('utf-8'))

# s3://gdit-faa-datachallenge-proto/notebooks/aria_airborne_pyspark_eda/
fig_filename = 'aria_airborne_events_plotly_heatmap_shader.html'
s3_client.put_object(Bucket='gdit-faa-datachallenge-proto', Key=f'notebooks/aria_airborne_pyspark_eda/{fig_filename}', Body=bytes_buf)
bytes_buf.close()
string_buf.close()

#### KMeans Clustering by Location

In [ ]:
combined_df = combined_df.drop('location_features')

In [ ]:
# TODO determine the optimal number of clusters (k)
k = 10 
kmeans = KMeans(k=k, featuresCol="location_features")

# Assemble features into a vector
assembler = VectorAssembler(inputCols=["latitude", "longitude"], outputCol="location_features")
combined_df = assembler.transform(combined_df)

# Train the model
kmeans_model = kmeans.fit(combined_df)

# Assign cluster labels to each track point
combined_df = kmeans_model.transform(combined_df)

In [ ]:
combined_df.show(n=1, vertical=True, truncate=True)

In [ ]:
combined_df.groupBy('prediction').count().show()

## Clean up Resources

In [ ]:
%stop_session